In [ ]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.animation as animation
%matplotlib inline

In [4]:
import sys  
sys.path.insert(1, '/Users/minliqiu/documents/research/dust/code/Hamiltonian/full_averaged_R2')

from disturb_Qavg_r3bp_helio import Q_averaged_disturb
from eq_points_r3bp_helio_conservative import lpe_r3bp_helio_kappa, root_find

$\gamma = \frac{1}{2}\dot{\Gamma}_{\Gamma} + \frac{1}{2}\dot{\kappa}_{\Gamma}\left(\frac{K_{0,\Gamma\kappa}}{K_{0,\Gamma\Gamma}}\right)$
--

$\kappa_2 = \frac{a_2}{a_{2, res}}\left(\frac{j\sqrt{1-e_2^2}-(j-k)}{k}\right)^2 - 1$

$\Gamma_2 = \sqrt{G m_0(1-\beta)a_2}(1-\sqrt{1-e_2^2})$


From the notes, the real part the eigenvalue for the third, non-oscillatory mode to the firt order is

$$\mathrm{Re}\left(s_3^{(1)}\right) = \dot{\kappa}_{\kappa} +
\dot{\kappa}_{\Gamma} \left[\frac{K_{\Gamma\phi}K_{\kappa\phi} - K_{\phi\phi}K_{\Gamma\kappa}}{\omega_0^2}\right]$$

where $\omega_0^2 = K_{\Gamma\Gamma}K_{\phi\phi} - K_{\Gamma\phi}^2$.

If the $\kappa$-dependence enters mainly through $K_0(\Gamma,\kappa)$, so that $K_{\kappa\phi}\simeq 0$, then

$$\mathrm{Re}\left(s_3^{(1)}\right) \simeq \dot{\kappa}_{\kappa} + \dot{\kappa}_{\Gamma} \frac{K_{\phi\phi}K_{0,\Gamma\kappa}}{\omega_0^2}.$$

And if additionally $K_{\Gamma\phi}^2\ll K_{\Gamma\Gamma}K_{\phi\phi}$, then

$$\mathrm{Re}\left(s_3^{(1)}\right) \simeq \dot{\kappa}_{\kappa} - \dot{\kappa}_{\Gamma} \left(\frac{K_{0,\Gamma\kappa}}{K_{0,\Gamma\Gamma}}\right).$$

**step 1**

$\dot{\kappa_2} = (1+\kappa_2)\left(\frac{\beta G m_0}{a_2^2 c}\right)\left[-\frac{2+3e_2^2}{(1-e_2^2)^{3/2}}+\frac{5je_2^2}{[j\sqrt{1-e_2^2}-j+k](1-e_2^2)}\right]$

$\dot{\kappa}_{\kappa}$ with $\Gamma_2$ fixed: $\dot{\kappa}_{\kappa} = \left(\frac{\partial \dot{\kappa}}{\partial \kappa_2}\right)_{e_2} + \left(\frac{\partial \dot{\kappa}}{\partial e_2}\right)_{\kappa_2}
\left(\frac{\partial e_2}{\partial \kappa_2}\right)_{\Gamma_2}$.

The second factor comes from holding $\Gamma_2(\kappa_2,e_2)$ fixed: $d\Gamma_2 = \frac{\partial \Gamma_2}{\partial \kappa_2}d\kappa_2+ \frac{\partial \Gamma_2}{\partial e_2}de_2 =0$,

so $\left(\frac{\partial e_2}{\partial \kappa_2}\right)_{\Gamma_2} =
\frac{(\partial \Gamma_2/\partial \kappa_2)_{e_2}}{(\partial \Gamma_2/\partial e_2)_{\kappa_2}}$

In [5]:
# Symbols
beta, G, m0, a2_res, c, j, k = sp.symbols("beta G m0 a2_res c j k")
kappa2, e2 = sp.symbols("kappa2 e2")

eta = sp.sqrt(1 - e2**2)

# a2(kappa2, e2)
F = (j*eta - (j-k))/k
a2 = (1 + kappa2) * a2_res / F**2

# Gamma2(kappa2, e2)
Gamma2_expr = sp.sqrt(G*m0*(1-beta)*a2) * (1 - eta)

# kappa_dot(kappa2, e2)
term1 = (2 + 3*e2**2) / (1 - e2**2)**sp.Rational(3, 2)
term2 = (5*j*e2**2 / ((j*sp.sqrt(1 - e2**2) - j + k) * (1 - e2**2)))
kappa2_dot = ((1 + kappa2) * beta*G*m0/(a2**2*c) * (-term1 + term2))

# derivatives
dGamma_de = sp.diff(Gamma2_expr, e2)
dGamma_dkappa = sp.diff(Gamma2_expr, kappa2)
de_dGamma_at_kappa = sp.simplify(1 / dGamma_de)
de_dkappa_at_Gamma = sp.simplify(-dGamma_dkappa / dGamma_de)
dkappadot_de = sp.diff(kappa2_dot, e2)
dkappadot_dkappa_at_e = sp.diff(kappa2_dot, kappa2)
kappadot_Gamma = sp.simplify(dkappadot_de * de_dGamma_at_kappa)
kappadot_kappa = sp.simplify(dkappadot_dkappa_at_e + dkappadot_de * de_dkappa_at_Gamma)

# Numerical evaluation
G_ = 6.6743e-11
c_ = 3e8
m_Star = 1.99e30
R_Sun = 6.957e8

a1 = 10 * R_Sun
beta_ = 0.1
j_ = 2.0
k_ = 1.0
n2_eq = 9.904864866756742e-06
e2_eq = 0.4811868255521742
a2_eq = (G_ * m_Star * (1 - beta_) / n2_eq**2)**(1/3)
a2_res_ = (a1* (j_/(j_ - k_))**(2/3)* (1 - beta_)**(1/3))
kappa2_eq = (a2_eq/a2_res_* (j_*np.sqrt(1 - e2_eq**2) - (j_ - k_))**2 / k_**2- 1)
Gamma2_eq = (np.sqrt(G_ * m_Star * (1 - beta_) * a2_eq) * (1 - np.sqrt(1 - e2_eq**2)))

subs_eq = {
    G: G_,
    c: c_,
    beta: beta_,
    j: j_,
    k: k_,
    m0: m_Star,
    a2_res: a2_res_,
    e2: e2_eq,
    kappa2: kappa2_eq,
}

kappadot_Gamma_num = sp.N(kappadot_Gamma.subs(subs_eq))
kappadot_kappa_num = sp.N(kappadot_kappa.subs(subs_eq))

print("Gamma2_eq =", Gamma2_eq)

print("kappadot_Gamma =", kappadot_Gamma_num)
print("kappadot_kappa =", kappadot_kappa_num)

Gamma2_eq = 139412885989976.53
kappadot_Gamma = 4.19622888357364e-24
kappadot_kappa = -5.14696479835075e-10


In [6]:
second_term_num = -1.57099265484624e-9
RE_s_3 = kappadot_kappa_num + second_term_num
print ('damping rate:', RE_s_3)

damping rate: -2.08568913468132e-9
